In [134]:
import os
import numpy as np

from typing import Iterable

import marisa_trie
import pydantic

from scipy.sparse import csc_matrix

In [135]:
def stream_docs(columnar_texts, token2id):
    for text in columnar_texts:
        tokens = text.lower().split()  # TODO: improve tokenization
        token_ids = np.array([token2id[token] for token in tokens if token in token2id], dtype=np.int32)
        yield token_ids

In [136]:
class Vocabulary:
    def __init__(self, trie: marisa_trie.Trie):
        self.trie = trie

    def __len__(self):
        return len(self.trie)

    def __contains__(self, token):
        return token in self.trie

    def id(self, token):
        return self.trie.get(token)

    def token(self, id):
        return self.trie.get(id)
    
    @property
    def n_vocab(self):
        return len(self.trie)

    @classmethod
    def from_token_set(cls, token_set: set) -> "Vocabulary":
        trie = marisa_trie.Trie(sorted(token_set))
        return cls(trie)


In [137]:
class FieldBasedColumnarTexts(Iterable):
    def __init__(self, documents: Iterable[dict], field_name: str, n_docs: int):
        self.documents = documents
        self.field_name = field_name
        self.n_docs = n_docs

    def __iter__(self):
        for doc in self.documents:
            yield doc[self.field_name]


class ColumnarStatistics(pydantic.BaseModel):
    field_name: str

    n_docs: int
    df: np.ndarray = pydantic.Field(
        default_factory=lambda: np.array([], dtype=np.int32))
    doc_len: np.ndarray = pydantic.Field(
        default_factory=lambda: np.array([], dtype=np.int32))
    idf: np.ndarray = pydantic.Field(
        default_factory=lambda: np.array([], dtype=np.float32))
    nnz_total: int = 0
    avg_doc_len: float = pydantic.Field(default=0.0)

    class Config:
        arbitrary_types_allowed = True


class ColumnarStatisticsBuilder:

    @classmethod
    def build(cls, field_columnar_texts: FieldBasedColumnarTexts, vocab: Vocabulary) -> ColumnarStatistics:
        if not vocab or not vocab.trie:
            raise ValueError("Vocabulary is not provided or invalid.")

        n_docs = field_columnar_texts.n_docs
        df = np.zeros(len(vocab), dtype=np.int32)
        doc_len = np.zeros(n_docs, dtype=np.int32)
        nnz_total = 0

        for d, terms in enumerate(stream_docs(field_columnar_texts, vocab.trie)):
            doc_len[d] = len(terms)
            uniq = np.unique(terms)
            df[uniq] += 1
            nnz_total += uniq.size

            if d > n_docs-1:
                break

        idf = np.log((n_docs - df + 0.5) / (df + 0.5))
        idf = np.maximum(idf, 0)
        avg_doc_len = doc_len.mean() if n_docs > 0 else 0.0

        return ColumnarStatistics(
            field_name=field_columnar_texts.field_name,
            n_docs=n_docs,
            df=df,
            doc_len=doc_len,
            idf=idf,
            nnz_total=nnz_total,
            avg_doc_len=avg_doc_len
        )

/tmp/ipykernel_253620/4276355058.py:12: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class ColumnarStatistics(pydantic.BaseModel):


In [ ]:
# _INDEX_DIR = "/path/to/index/dir"
_INDEX_DIR = "./path/to/index/dir"
_INDEX_FILE = "indices.bin"
_DATA_FILE  = "data.bin"


class TermScore:
    term: str
    score: float

class DocumentScore:
    field_name: str
    doc_id: int
    term_scores: list[TermScore]

class BM25Index:
    field_name: str
    index: csc_matrix
    vocab: Vocabulary

    def get_score(self, doc_id: int, terms: list[str]) -> DocumentScore:
        if not self.vocab or not self.vocab.trie:
            raise ValueError("Vocabulary is not set in BM25Index.")

        trie = self.vocab.trie

        if doc_id < 0 or doc_id >= self.index.shape[0]:
            raise ValueError(f"Document ID {doc_id} not found in index.")

        term_scores = []
        for term in terms:
            if term in trie:
                score = float(self.index[doc_id, trie[term]])
                term_score = TermScore(term, score)
                term_scores.append(term_score)
        return DocumentScore(doc_id, term_scores)


class BM25IndexBuilder:
    @classmethod
    def load(cls, field_name: str, columnar_posting: Iterable[str], vocab: Vocabulary, field_statistics: ColumnarStatistics, index_file: str=_INDEX_FILE, data_file: str=_DATA_FILE) -> BM25Index:

        os.makedirs(_INDEX_DIR, exist_ok=True)
        index_filepath = os.path.join(_INDEX_DIR, index_file)
        data_filepath  =  os.path.join(_INDEX_DIR, data_file)

        k1, b = 1.5, 0.75  # BM25 parameters
        n_docs  = field_statistics.n_docs
        avg_doc_len = field_statistics.avg_doc_len
        df = field_statistics.df
        idf_of_terms = field_statistics.idf
        doc_len_of_docs = field_statistics.doc_len
        nnz_total = field_statistics.nnz_total
        n_vocab = len(vocab)

        indptr = np.empty(n_vocab + 1, dtype=np.int32)
        indptr[0] = 0
        np.cumsum(df, out=indptr[1:])

        indices = np.memmap(index_filepath, dtype=np.int32, mode="w+", shape=(nnz_total,))
        data    = np.memmap(data_filepath,  dtype=np.float32, mode="w+", shape=(nnz_total,))
        offset = indptr.copy()

        for d, terms in enumerate(stream_docs(columnar_posting, vocab.trie)):
            uniq, counts = np.unique(terms, return_counts=True)
            dl = float(doc_len_of_docs[d])
            tf = counts.astype(np.float32)

            for term, freq in zip(uniq.astype(np.int32), tf):
                idf = idf_of_terms[term]
                denom = freq + k1 * (1 - b + b * (dl / avg_doc_len))
                score = idf * (freq * (k1 + 1)) / denom

                pos = offset[term]
                indices[pos] = d
                data[pos] = score
                offset[term] += 1

        index   = csc_matrix((data, indices, indptr), shape=(n_docs, n_vocab))

        bm25_index = BM25Index()
        bm25_index.field_name = field_name
        bm25_index.index = index
        bm25_index.vocab = vocab
        return bm25_index

# Test

In [139]:
# Example documents
documents = [
    {"title": "Cat Facts", "text": "Cats are curious animals."},
    {"title": "Dog Facts", "text": "Dogs are loyal and friendly."},
    {"title": "Bird Facts", "text": "Birds can fly and sing."}
]

n_docs = len(documents)

# Build vocabulary and trie
all_tokens = set()
for doc in documents:
    all_tokens.update(doc["text"].lower().split())


## Build BM25 Index of 'text' field of document

In [140]:
field_name = "text"

In [141]:
field_columnar_texts = FieldBasedColumnarTexts(documents, field_name, n_docs)

In [142]:
vocab: Vocabulary = Vocabulary.from_token_set(all_tokens)

In [143]:
statistics: ColumnarStatistics = ColumnarStatisticsBuilder.build(field_columnar_texts, vocab)

In [144]:
statistics

ColumnarStatistics(field_name='text', n_docs=3, df=array([1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1], dtype=int32), doc_len=array([4, 5, 5], dtype=int32), idf=array([0.51082562, 0.51082562, 0.51082562, 0.51082562, 0.        ,
       0.51082562, 0.51082562, 0.51082562, 0.        , 0.51082562,
       0.51082562, 0.51082562]), nnz_total=14, avg_doc_len=4.666666666666667)

In [145]:
bm25_index: BM25Index = BM25IndexBuilder.load(field_name, field_columnar_texts, vocab, statistics)

indptr: [ 0  1  2  3  4  6  7  8  9 11 12 13 14]
(13,)
offset samples: [ 0  1  2  3  4  6  7  8  9 11]
